In [10]:
### dataset class to work with images and labels
# takes an excel sheet as input with the columns 'path' and 'label'

from torch.utils.data import Dataset, Subset
from PIL import Image
import pandas as pd
import os

class ImagePathDataset(Dataset):
    """
    Dataset that reads image paths and labels from an Excel sheet.
    Optionally replaces old drive roots (e.g., D:/, E:/) with a new root directory.
    """
    def __init__(
        self,
        excel_file,
        img_col="path",
        label_col="label",
        transform=None,
        new_root=None,
        old_roots=("D:/", "E:/backup/")
    ):
        """
        Args:
            excel_file (str): Path to Excel file containing image paths and labels.
            img_col (str): Name of the column containing image paths.
            label_col (str): Name of the column containing labels.
            transform (callable, optional): Optional torchvision transform.
            new_root (str, optional): Replace old root (e.g., 'D:/') with this new root.
            old_roots (tuple, optional): Roots to replace, e.g. ('D:/', 'E:/').
        """
        self.data = pd.read_excel(excel_file)
        self.img_col = img_col
        self.label_col = label_col
        self.transform = transform
        self.new_root = new_root
        self.old_roots = old_roots

    def __len__(self):
        return len(self.data)

    def _replace_root(self, path):
        """Replace old drive roots with the new root if specified."""
        if self.new_root is None:
            return path

        # Normalize slashes for consistency
        path = path.replace("\\", "/")
        for old_root in self.old_roots:
            old_root = old_root.replace("\\", "/")
            if path.startswith(old_root):
                rel_path = path[len(old_root):].lstrip("/")
                return os.path.normpath(os.path.join(self.new_root, rel_path))
        # If no match, return path unchanged
        return path

    def __getitem__(self, idx):
        img_path = self.data.loc[idx, self.img_col]
        label = self.data.loc[idx, self.label_col]

        img_path = self._replace_root(img_path)

        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)

        return image, label

3156
[]


In [ ]:
### datasplitter: makes sure the labels are balanced in all subsets

from collections import defaultdict
import random

def stratified_split(dataset, val_ratio, test_ratio, seed=42):
    random.seed(seed)
    
    labels = [dataset[i][1] for i in range(len(dataset))]
    label_to_indices = defaultdict(list)
    for idx, label in enumerate(labels):
        label_to_indices[label].append(idx)
    
    train_indices = []
    val_indices = []
    test_indices = []
    
    for label, indices in label_to_indices.items():
        random.shuffle(indices)
        n = len(indices)
        n_test = int(n * test_ratio)
        n_val = int(n * val_ratio)
    
        test_indices.extend(indices[:n_test])
        val_indices.extend(indices[n_test:n_test + n_val])
        train_indices.extend(indices[n_test + n_val:])
    
    train_dataset = Subset(dataset, train_indices)
    val_dataset = Subset(dataset, val_indices)
    test_dataset = Subset(dataset, test_indices)

    return train_dataset, val_dataset, test_dataset

In [ ]:
### example workflow

# create dataset
dataset = ImagePathDataset('path/to/excel/sheet.xlsx', new_root='D:/')    # new_root should be the interface where the hard drive is connected (in the sheet all paths start with E:/backup/...)

# split dataset
train_dataset, val_dataset, test_dataset = stratified_split(dataset, 0.15, 0.15, seed=42)

# or to test, just
_, _, test_dataset = stratified_split(dataset, 0.15, 0.15, seed=42)
